# 🐾 Módulo 02 — Ambiente do Quadrúpede Anymal C

**Duração:** 30 minutos (0:55 – 1:25) | **Workshop Quadrúpede — Summit IA Joinville**

### 🎯 Objetivos
1. ✅ Conhecer o robô **Anymal C** (hardware e cinemática)
2. ✅ Entender a **formulação MDP** completa
3. ✅ Detalhar o **vetor de observação** (48D)
4. ✅ Entender o **espaço de ações** (12 juntas)
5. ✅ Analisar a **função de recompensa** multi-objetivo
6. ✅ Criar e inspecionar o ambiente `Workshop-Anymal-v0`


## 🤖 O Robô Anymal C

```
              ╔══════════════════╗
              ║    ANYMAL C      ║
              ╚══════════════════╝

    LF (Left Front)      RF (Right Front)
      🦵                    🦵
       \                   /
        ●─────[BODY]─────●
       /    ~30kg corpo    \
      🦵                   🦵
    LH (Left Hind)       RH (Right Hind)

  Cada pata:
  HAA (Hip Abduction/Adduction) ──▶ lateral
  HFE (Hip Flexion/Extension)   ──▶ frente/trás
  KFE (Knee Flexion/Extension)  ──▶ joelho
```

| Característica | Valor |
|---|---|
| Origem | ETH Zurich / ANYbotics |
| Massa | ~30 kg |
| Graus de liberdade | 12 DOF (3 × 4 patas) |
| Tipo de juntas | Torque-controlled |
| Aplicações | Inspeção industrial, busca e resgate |


## 🔄 Formulação MDP

```
┌──────────────────────────────────────────────────┐
│              CICLO MDP — Anymal Workshop          │
│                                                  │
│  Isaac Sim ──▶ obs (48D) ──▶ Política (MLP)     │
│      ▲                           │               │
│      └────── ação (12D) ◀────────┘               │
│                                                  │
│  sim.step() → sₜ₊₁, rₜ, done                    │
│  Episode: 1000 steps × 20ms = 20 segundos        │
│  Num. envs: 4096 (treino) / 16 (debug)           │
└──────────────────────────────────────────────────┘
```

**Tarefa:** o robô recebe comando de velocidade `[vx, vy, wz]` e deve seguir.
- `vx` ∈ [-1.0, 1.0] m/s — frente/trás
- `vy` ∈ [-0.5, 0.5] m/s — lateral
- `wz` ∈ [-1.0, 1.0] rad/s — rotação


## 👁️ Vetor de Observação (48 dimensões)

| Índices | Componente | Dimensões | Intuição |
|---------|------------|-----------|----------|
| [0:3]   | Vel. linear do corpo | 3 | Quão rápido se move |
| [3:6]   | Vel. angular do corpo | 3 | Quão rápido gira |
| [6:9]   | Gravidade projetada | 3 | Para onde é 'baixo' → equilíbrio |
| [9:12]  | Comando velocidade | 3 | O que o operador quer |
| [12:24] | Posição juntas (rel.) | 12 | Onde cada junta está |
| [24:36] | Velocidade juntas | 12 | Quão rápido se move |
| [36:48] | Ação anterior | 12 | Para suavidade do movimento |
| — | **Total** | **48** | |

```python
obs = torch.cat([
    base_lin_vel_b,                    # [3]
    base_ang_vel_b,                    # [3]
    projected_gravity_b,               # [3]
    commands,                          # [3]
    joint_pos - default_joint_pos,     # [12]
    joint_vel,                         # [12]
    last_action,                       # [12]
], dim=-1)  # total = 48
```


## 🎮 Espaço de Ações (12 dimensões)

A política produz **position residuals** (desvios da posição padrão):

```
Política → aₜ ∈ [-1, 1]¹²
q_target = q_default + 0.5 × aₜ  (rad)
τ = Kp × (q_target - q) + Kd × (-q̇)  ← PD controller (200 Hz)
```

| Índice | Junta | Descrição |
|--------|-------|-----------|
| 0 | LF_HAA | Quadril esq. frente (ab/adução) |
| 1 | LF_HFE | Quadril esq. frente (flex/ext) |
| 2 | LF_KFE | Joelho esq. frente |
| 3-5 | RF_* | Pata direita frente |
| 6-8 | LH_* | Pata esquerda traseira |
| 9-11 | RH_* | Pata direita traseira |

> 💡 A política **não** controla torques diretamente — controla posição alvo. O PD faz a conversão.


## 🏆 Função de Recompensa

### ✅ Termos Positivos:
| Termo | Peso | Fórmula |
|-------|------|---------|
| `track_lin_vel_xy` | +1.0 | `exp(-‖v_atual - v_cmd‖²/0.25)` |
| `track_ang_vel_z` | +0.5 | `exp(-‖ω_atual - ω_cmd‖²/0.25)` |
| `feet_air_time` | +0.25 | Incentivar passos definidos |

### ❌ Termos Negativos (Penalidades):
| Termo | Peso | Intuição |
|-------|------|----------|
| `lin_vel_z` | -2.0 | Não saltar/afundar |
| `ang_vel_xy` | -0.05 | Não rolar/tombar |
| `flat_orientation` | -5.0 | Manter corpo horizontal |
| `joint_acc` | -2.5e-7 | Movimento suave |
| `action_rate` | -0.01 | Ações suaves |

> 💡 **Design da reward function é arte + ciência!** Os pesos controlam o comportamento emergente.


In [ ]:
# 📄 Mostrar configuração do ambiente
import inspect, os

# Tentar importar a configuração real
try:
    from workshop_quadrupede.envs import AnymalWorkshopEnvCfg
    print('✅ AnymalWorkshopEnvCfg carregada!')
    print(f'  action_space:      {AnymalWorkshopEnvCfg.action_space}')
    print(f'  observation_space: {AnymalWorkshopEnvCfg.observation_space}')
    print(f'  num_envs (padrão): {AnymalWorkshopEnvCfg.scene.num_envs}')
    print(f'  episode_length_s:  {AnymalWorkshopEnvCfg.episode_length_s}')
    print(f'  sim dt:            {AnymalWorkshopEnvCfg.sim.dt}s ({1/AnymalWorkshopEnvCfg.sim.dt:.0f} Hz)')
except ImportError:
    print('⚠️  workshop_quadrupede não está importável neste kernel.')
    print('   Execute com: ./isaaclab.sh -p workshop/scripts/treinar.py')
    print()
    print('Configuração esperada:')
    print('  action_space:      12  (12 joint targets)')
    print('  observation_space: 48  (48D vector)')
    print('  num_envs (padrão): 4096')
    print('  episode_length_s:  20.0s (1000 steps × 20ms)')
    print('  sim dt:            0.005s (200 Hz)')


In [ ]:
# 📊 Script para criar o ambiente (rode via CLI)
script = '''
# Execute: ./isaaclab.sh -p criar_env.py
from isaaclab.app import AppLauncher
app = AppLauncher(headless=True)
sim = app.app

import gymnasium as gym
import workshop_quadrupede  # registra Workshop-Anymal-v0
from isaaclab_tasks.utils import parse_env_cfg

env_cfg = parse_env_cfg('Workshop-Anymal-v0', device='cuda', num_envs=16)
env = gym.make('Workshop-Anymal-v0', cfg=env_cfg)

print(f'Num envs:     {env.unwrapped.num_envs}')
print(f'Obs space:    {env.observation_space}')
print(f'Action space: {env.action_space}')

obs, info = env.reset()
print(f'Obs shape:  {obs["policy"].shape}')  # (16, 48)
env.close()
sim.close()
'''
print('📄 Script de criação do ambiente:')
print(script)

print('📊 Saída esperada:')
print('  Num envs:     16')
print('  Obs space:    Box(-inf, inf, (48,), float32)')
print('  Action space: Box(-1.0, 1.0, (12,), float32)')
print('  Obs shape:    torch.Size([16, 48])')


In [ ]:
# 📊 Análise da Função de Recompensa
import numpy as np

print('📊 Análise: Agente ALEATÓRIO vs TREINADO')
print('=' * 60)

termos = [
    ('track_lin_vel_xy',   +1.000, 0.05, 0.87),
    ('track_ang_vel_z',    +0.500, 0.10, 0.75),
    ('lin_vel_z',          -2.000, -0.45, -0.02),
    ('ang_vel_xy',         -0.050, -0.38, -0.05),
    ('flat_orientation',   -5.000, -0.30, -0.02),
    ('joint_acc',          -2.5e-7, -0.15, -0.08),
    ('action_rate',        -0.010, -0.25, -0.04),
]

print(f'  {"Termo":<25} {"Peso":>8}  {"Aleatório":>10}  {"Treinado":>10}')
print('  ' + '-' * 58)

total_al = total_tr = 0
for nome, peso, val_al, val_tr in termos:
    ca = peso * val_al
    ct = peso * val_tr
    total_al += ca
    total_tr += ct
    sinal = '🟢' if peso > 0 else '🔴'
    print(f'  {sinal} {nome:<23} {peso:>8.4f}  {ca:>10.4f}  {ct:>10.4f}')

print('  ' + '-' * 58)
print(f'  {"TOTAL":>32}  {total_al:>10.4f}  {total_tr:>10.4f}')
print(f'\n💡 Melhora: {total_tr/abs(total_al):.1f}x mais recompensa após treino!')
print('\n➡️  Próximo: Módulo 03 — Treinar com RSL-RL!')
